# Maze Crawler - Submit *Harvester v2*

Create from the competition page so `crawl` is available. **Run all**, then **Save Version** and **Submit** the produced `submission.tar.gz` (or `main.py`).

v2 improves on v1's survival (reactive jumping to bypass walls, never steps south, escapes when buffer is thin) and adds an economy layer (scouts harvest crystals and bank energy into the uncapped factory). Locally it beats v1 ~12-1-1 and goes 8/8 vs the built-in `random` agent.

In [1]:
%%writefile main.py
"""
Maze Crawler - Harvester v2
===========================
v1's PROVEN survival logic (factory unchanged) + an economy layer on the units:
  * Scouts actively seek nearby crystals, then dump energy into the uncapped
    factory via TRANSFER when adjacent -> grows the factory's tiebreaker hoard.
  * Workers still clear walls / rescue the factory, and also bank spare energy.
The factory keeps marching north exactly as in v1 (do not regress survival).

`agent` MUST remain the LAST function. Types 0=F 1=S 2=W 3=M. Walls N1 E2 S4 W8.
robot = [type,col,row,energy,owner,move_cd,jump_cd,build_cd]
"""

from collections import deque

STATE = {"step": -1, "last_south": None}
DIRS = {"NORTH": (0, 1), "SOUTH": (0, -1), "EAST": (1, 0), "WEST": (-1, 0)}
DIRBIT = {"NORTH": 1, "EAST": 2, "SOUTH": 4, "WEST": 8}
TO_FACTORY = {(0, 1): "TRANSFER_NORTH", (0, -1): "TRANSFER_SOUTH",
              (1, 0): "TRANSFER_EAST", (-1, 0): "TRANSFER_WEST"}
FACTORY, SCOUT, WORKER, MINER = 0, 1, 2, 3
TARGET_SCOUTS = 2
TARGET_WORKERS = 2


def _cfg(config, name, default):
    try:
        v = getattr(config, name)
        if v is not None:
            return v
    except Exception:
        pass
    try:
        return config[name]
    except Exception:
        return default


def _obs(obs, name, default):
    try:
        v = getattr(obs, name)
        if v is not None:
            return v
    except Exception:
        pass
    try:
        return obs[name]
    except Exception:
        return default


def _bfs_first_step(start, can_step):
    visited = {start}
    firstmove = {start: None}
    q = deque([start])
    best, best_row = start, start[1]
    while q:
        c, r = q.popleft()
        if r > best_row:
            best_row, best = r, (c, r)
        for dirn, (dx, dy) in DIRS.items():
            if not can_step(c, r, dirn):
                continue
            nxt = (c + dx, r + dy)
            if nxt in visited:
                continue
            visited.add(nxt)
            firstmove[nxt] = firstmove[(c, r)] if firstmove[(c, r)] is not None else dirn
            q.append(nxt)
    if best_row <= start[1]:
        return None
    return firstmove.get(best)


def _decide(obs, config):
    W = int(_cfg(config, "width", 20))
    walls = list(_obs(obs, "walls", []))
    south = int(_obs(obs, "southBound", 0))
    north = int(_obs(obs, "northBound", south + (len(walls) // W if W else 20) - 1))
    me = _obs(obs, "player", 0)
    robots = _obs(obs, "robots", {})
    crystals = _obs(obs, "crystals", {}) or {}
    episode_steps = int(_cfg(config, "episodeSteps", 501))
    scout_cost = int(_cfg(config, "scoutCost", 50))
    worker_cost = int(_cfg(config, "workerCost", 200))
    remove_cost = int(_cfg(config, "wallRemoveCost", 100))

    last_south = STATE.get("last_south")
    if last_south is not None and south < last_south - 2:
        STATE["step"] = -1
    STATE["step"] += 1
    STATE["last_south"] = south
    step = STATE["step"]
    turns_left = max(20, episode_steps - step)

    def wbits(col, row):
        if not (0 <= col < W):
            return None
        i = (row - south) * W + col
        if 0 <= i < len(walls):
            v = walls[i]
            return None if v == -1 else v
        return None

    def on_board(col, row):
        return 0 <= col < W and south <= row <= north

    def can_step(col, row, dirn):
        b = wbits(col, row)
        if b is None:
            return False
        if b & DIRBIT[dirn]:
            return False
        dx, dy = DIRS[dirn]
        return on_board(col + dx, row + dy)

    my_units = {}
    enemy_cells, enemy_factory_cells = set(), set()
    for uid, d in robots.items():
        if d[4] == me:
            my_units[uid] = d
        else:
            enemy_cells.add((d[1], d[2]))
            if d[0] == FACTORY:
                enemy_factory_cells.add((d[1], d[2]))

    factory_uid = factory = None
    scouts, workers, miners = [], [], []
    for uid, d in my_units.items():
        if d[0] == FACTORY:
            factory_uid, factory = uid, d
        elif d[0] == SCOUT:
            scouts.append((uid, d))
        elif d[0] == WORKER:
            workers.append((uid, d))
        elif d[0] == MINER:
            miners.append((uid, d))

    actions, claimed, friendly_final = {}, set(), set()
    factory_pos = (factory[1], factory[2]) if factory else None
    factory_energy = factory[3] if factory else 0
    fcol = factory_pos[0] if factory_pos else W // 2

    def crystal_at(col, row):
        return ("%d,%d" % (col, row)) in crystals

    def adj_factory_dir(col, row):
        if factory_pos is None:
            return None
        return TO_FACTORY.get((factory_pos[0] - col, factory_pos[1] - row))

    def nearest_crystal(col, row, max_dist=8):
        best, bestscore = None, None
        for key, e in crystals.items():
            try:
                cc, cr = key.split(","); cc, cr = int(cc), int(cr)
            except Exception:
                continue
            if cr < south + 2:
                continue
            d = abs(cc - col) + abs(cr - row)
            if d == 0 or d > max_dist:
                continue
            sc = d - e * 0.03
            if bestscore is None or sc < bestscore:
                bestscore, best = sc, (cc, cr)
        return best

    def toward(col, row, target, fallback):
        pref = []
        if target:
            tc, tr = target
            if tr > row: pref.append("NORTH")
            if tc > col: pref.append("EAST")
            if tc < col: pref.append("WEST")
            if tr < row: pref.append("SOUTH")
        for d in fallback:
            if d not in pref:
                pref.append(d)
        return pref

    def pick_move(col, row, prefer, avoid_enemy=True, allow_south=False, col_bias=None):
        order, seen = [], set()
        for d in prefer:
            if d not in seen and (allow_south or d != "SOUTH"):
                seen.add(d); order.append(d)
        best, best_score = None, None
        for dirn in order:
            if not can_step(col, row, dirn):
                continue
            dx, dy = DIRS[dirn]
            nc, nr = col + dx, row + dy
            cell = (nc, nr)
            if cell in claimed:
                continue
            if avoid_enemy and cell in enemy_cells:
                continue
            score = order.index(dirn)
            if crystal_at(nc, nr):
                score -= 100
            if col_bias is not None:
                score += abs(nc - col_bias) * 0.01
            if best_score is None or score < best_score:
                best_score, best = score, (dirn, cell)
        return best if best else ("IDLE", (col, row))

    # ---------- SCOUTS: harvest crystals, bank into factory ----------
    for i, (uid, d) in enumerate(scouts):
        col, row, energy = d[1], d[2], d[3]
        tdir = adj_factory_dir(col, row)
        if tdir and energy >= 40:
            actions[uid] = tdir
            friendly_final.add((col, row)); claimed.add((col, row)); continue
        if energy <= 0:
            friendly_final.add((col, row)); claimed.add((col, row)); continue
        if i % 2 == 0:
            npref, bias = ["NORTH", "EAST", "WEST"], min(W - 1, col + 4)
        else:
            npref, bias = ["NORTH", "WEST", "EAST"], max(0, col - 4)
        tgt = nearest_crystal(col, row, 8)
        allow_s = tgt is not None and tgt[1] >= south + 2 and tgt[1] < row
        pref = toward(col, row, tgt, npref)
        act, cell = pick_move(col, row, pref, allow_south=allow_s, col_bias=bias)
        actions[uid] = act
        friendly_final.add(cell); claimed.add(cell)

    # ---------- WORKERS: clear walls / rescue factory / harvest ----------
    for uid, d in workers:
        col, row, energy = d[1], d[2], d[3]
        b = wbits(col, row)
        buf = row - south
        # rescue: feed a starving factory
        if factory_pos is not None and factory_energy < 150 and energy > 20:
            tdir = adj_factory_dir(col, row)
            if tdir:
                actions[uid] = tdir
                friendly_final.add((col, row)); claimed.add((col, row)); continue
        # clear a north wall ahead (helps the factory's lane)
        if (b is not None) and (b & 1) and energy > remove_cost + 5 and row < north and buf >= 1:
            actions[uid] = "REMOVE_NORTH"
            friendly_final.add((col, row)); claimed.add((col, row)); continue
        # bank spare energy when adjacent
        tdir = adj_factory_dir(col, row)
        if tdir and energy >= 120:
            actions[uid] = tdir
            friendly_final.add((col, row)); claimed.add((col, row)); continue
        tgt = nearest_crystal(col, row, 5)
        pref = toward(col, row, tgt, ["NORTH", "EAST", "WEST"])
        act, cell = pick_move(col, row, pref, col_bias=fcol)
        actions[uid] = act
        friendly_final.add(cell); claimed.add(cell)

    for uid, d in miners:
        col, row = d[1], d[2]
        act, cell = pick_move(col, row, ["NORTH", "EAST", "WEST"], col_bias=fcol)
        actions[uid] = act
        friendly_final.add(cell); claimed.add(cell)

    # ---------- FACTORY: identical to v1 (proven survival) ----------
    if factory is not None:
        col, row, energy = factory[1], factory[2], factory[3]
        jump_cd = factory[6] if len(factory) > 6 else 0
        build_cd = factory[7] if len(factory) > 7 else 0
        fb = wbits(col, row)
        north_wall = (fb is not None) and (fb & 1)
        buffer = row - south
        early_econ = step < 130

        def north_cell_safe():
            nc = (col, row + 1)
            if nc in enemy_factory_cells:
                return False
            if nc in friendly_final:
                return False
            return True

        def jump_ok():
            return jump_cd == 0 and (row + 2) <= north

        n_scouts = len(scouts)
        n_workers = len(workers)
        spawn_free = (not north_wall) and (row + 1 <= north) \
            and ((col, row + 1) not in friendly_final) \
            and ((col, row + 1) not in enemy_cells)
        build_action = None
        if build_cd == 0 and spawn_free and early_econ and buffer >= 3:
            if n_scouts < TARGET_SCOUTS and (energy - scout_cost) > (turns_left + 60):
                build_action = "BUILD_SCOUT"
            elif n_workers < TARGET_WORKERS and (energy - worker_cost) > (turns_left + 60):
                build_action = "BUILD_WORKER"

        def safe(cell):
            return cell not in friendly_final and cell not in enemy_factory_cells \
                and cell not in enemy_cells

        def northward_move():
            jr = jump_ok()
            # Jump to bypass a blocking wall, or proactively when buffer is thin
            # or in the fast-scroll endgame; otherwise SAVE the jump for walls.
            if jr and (north_wall or buffer <= 8 or step > 430):
                return "JUMP_NORTH"
            if not north_wall and (row + 1 <= north) and north_cell_safe() \
               and (col, row + 1) not in enemy_cells:
                return "NORTH"
            if jr:  # north blocked, jump is our bypass
                return "JUMP_NORTH"
            # route around using BFS, but NEVER step south (toward the boundary)
            fm = _bfs_first_step((col, row), can_step)
            if fm is not None and fm != "SOUTH":
                dx, dy = DIRS[fm]
                if safe((col + dx, row + dy)):
                    return fm
            for dirn in ("EAST", "WEST"):
                if can_step(col, row, dirn):
                    dx, dy = DIRS[dirn]
                    cell = (col + dx, row + dy)
                    if safe(cell) and cell not in claimed:
                        return dirn
            return "IDLE"

        actions[factory_uid] = build_action if build_action is not None else northward_move()

    return actions


def agent(obs, config):
    try:
        return _decide(obs, config)
    except Exception:
        actions = {}
        try:
            me = _obs(obs, "player", 0)
            for uid, d in _obs(obs, "robots", {}).items():
                if d[4] == me:
                    actions[uid] = "BUILD_WORKER" if d[0] == FACTORY else "NORTH"
        except Exception:
            pass
        return actions


Writing main.py


In [ ]:
# Validate: full mirror match (mimics Kaggle's validation episode)
from kaggle_environments import make
env = make('crawl', configuration={'randomSeed': 42}, debug=True)
env.run(['main.py', 'main.py'])
st = [s.status for s in env.steps[-1]]
print('len', len(env.steps), 'rewards', [s.reward for s in env.steps[-1]], 'status', st)
assert 'ERROR' not in st and 'INVALID' not in st, 'AGENT ERRORED'
print('OK: no errors.')

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment

In [ ]:
# Signal: v2 vs the built-in random agent, both sides
wins=0
for s in range(6):
    env=make('crawl',configuration={'randomSeed':s}); env.run(['main.py','random'])
    r=env.steps[-1]; wins += (r[0].reward is not None and r[0].reward> (r[1].reward or 0))
for s in range(6):
    env=make('crawl',configuration={'randomSeed':100+s}); env.run(['random','main.py'])
    r=env.steps[-1]; wins += (r[1].reward is not None and r[1].reward> (r[0].reward or 0))
print(f'v2 vs random: {wins}/12 wins')

In [ ]:
# Watch a game (v2 = player 0)
env=make('crawl',configuration={'randomSeed':1}); env.run(['main.py','random'])
env.render(mode='ipython', width=720, height=720)

In [ ]:
import tarfile
with tarfile.open('/kaggle/working/submission.tar.gz','w:gz') as t:
    t.add('main.py', arcname='main.py')
print('Wrote submission.tar.gz - Save Version, then Submit on the competition page.')